In [1]:
from pathlib  import Path
import pandas as pd
import matplotlib as plt
import os
import sys
import json
import nltk
from nltk import FreqDist
import matplotlib.pyplot as plt
from unidecode import unidecode
from gensim.parsing.preprocessing import split_alphanum
from gensim.parsing.preprocessing import strip_non_alphanum
from gensim.parsing.preprocessing import strip_punctuation
from gensim.parsing.preprocessing import strip_tags
from gensim.parsing.preprocessing import strip_multiple_whitespaces
sys.path.append('../')
import spacy
nlp = spacy.load("fr_core_news_sm")
import numpy as np

In [2]:
path_name = '/home/ml/BI_Help/data/raw/Fusion_Procurement_BI/'
path_file_xls = '/home/ml/BI_Help/data/raw/Import/corresp_Labelle/text_material_group_AVS_FR.xlsx'
path_file_text = '/home/ml/BI_Help/data/raw/Import/corresp_Labelle/MATKL T023T ALL AQ6.txt'

In [3]:
path_file_pkl ='/home/ml/BI_Help/data/processed/codes_labels.pkl'
path_code_corrige = '/home/ml/BI_Help/data/processed/codes_corrige.pkl'

In [4]:
EKPO_files = sorted(Path(path_name).glob("*EKPO*"))

In [5]:
def merge_total():
    df_list = []
    for i in range(len(EKPO_files)):
        df_list.append(pd.read_csv(str(EKPO_files[i]),sep = ';', error_bad_lines=False))
    return pd.concat(df_list)

def lemma(x):
    doc = nlp(x)
    tokens = [token.text for token in doc]
    list_test = []
    for token in doc:
        list_test.append(token.lemma_)
    return list_test

def correction(row):
    if 'utiliser' in str(row['Long']):
        return row['Long'].replace('utiliser ','').strip()
    else:
        return row['MATKLMaterialGroup']

In [6]:
df_total = merge_total()

/home/ml/.local/lib/python3.5/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning: Columns (8,11,12,17,22,25,27,35,59,61,73,78,95,101,109,161,198) have mixed types. Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/home/ml/.local/lib/python3.5/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning: Columns (8,10,11,17,20,22,25,27,35,59,60,61,72,73,78,95,101,109,160,161,198) have mixed types. Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/home/ml/.local/lib/python3.5/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning: Columns (1,10,11,12,17,20,22,25,27,35,60,61,72,73,78,95,101,109,160,161,198) have mixed types. Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
/home/ml/.local/lib/python3.5/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning: Columns

In [7]:
df_correction = pd.read_pickle(path_code_corrige)

In [8]:
df_correction

,Matl Group,code_correction
1092,01000,01200
1098,015000,02800
1099,015001,02800
1109,025001,02300A
1110,025002,02300B
...,...,...
1675,95200,36100
1677,96100,95000
1678,97100,95000
1679,9800A,Ne plus


In [9]:
df_total = df_total[['TXZ01 Short Text' , 'MATKL Material Group']]
df_total.drop_duplicates(inplace=True)

In [10]:
df_correction.rename(columns = {'Matl Group' : 'MATKL Material Group'}, inplace=True)

In [11]:
df_merge = pd.merge(df_total, df_correction, on='MATKL Material Group', how='left')

In [13]:
def correction_test(row):
    
    if row.notnull()['code_correction']:
        return row['code_correction']
    else :
        return row['MATKL Material Group']

In [14]:
df_merge['code_correction'] = df_merge[['MATKL Material Group', 'code_correction']].apply(correction_test, axis=1)

In [15]:
def df_malk_correction():
    df_malk = pd.read_pickle(path_file_pkl)
    df_malk['Matl Group'] = df_malk['Matl Group'].astype(str)
    df_malk['Material grp desc. 2'] = df_malk['Material grp desc. 2'].astype(str)
    def correction(row):
        if 'utiliser' in str(row['Material grp desc. 2']):
            return row['Material grp desc. 2'].replace('utiliser ', '').strip()
        else:
            return row['Matl Group']
    df_malk['code_correction'] = df_malk[['Material grp desc. 2', 'Matl Group']].apply(correction, axis=1)
    df_malk.dropna(inplace=True)
    return df_malk[~df_malk['Material grp desc. 2'].str.contains("utiliser")]

In [16]:
df_malk = df_malk_correction()

In [19]:
df_util = pd.merge(df_merge, df_malk[['code_correction', 'Material grp desc. 2']], on = 'code_correction', how = 'left')

In [20]:
df_util

,TXZ01 Short Text,MATKL Material Group,code_correction,Material grp desc. 2
0,PIGNON 36 DENTS,71100,42500,"Comp. Méca. (moteur, actionneur, vérin, roulem..."
1,VIS,NDF00,NDF00,CTA A CHANGER ...
2,"GAINE RETRACTABLE DIAM.1,2MM TRANSPARENT",86100,42500,"Comp. Méca. (moteur, actionneur, vérin, roulem..."
3,ACC_CON PROTECTEUR CARTE EMB,44100,44100,connecteurs circulaires et accessoires ...
4,CONNECTEUR HE804ES53Z1A221,44200,44100,connecteurs circulaires et accessoires ...
...,...,...,...,...
80938,SILOUET - Data management,60600,60600,Prest en AT Ing Syst Hors IVVQ ...
80939,Resistance A2022977,NDF00,NDF00,CTA A CHANGER ...
80940,GDM V3 Livraison en recette,15201A,15201A,ST Dév d'appli ou infra interne (Fnr hors Gpe)...
80941,GDM V3 Mise en production,15201A,15201A,ST Dév d'appli ou infra interne (Fnr hors Gpe)...
